<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_09_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

untill now one attention mechanism but transformers ask why use only one way to understand sentence?
instead use multiple attention heads each head learns different patterns

e.g 'the cat sat on a mat'

one person focus on grammer
another on meaning
another on object relationship

transformer does same thing

**why important ?**

single attention imited understanding

multi head multiple perspective simultaneously

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F



**Architecture**

input sentence : i love NLP

this sentence goes into multiple attenstion suppose in head 1, head 2, head 3, head4. each head creates diffrent attention map and different understanding then combine all outputs

and each head has different query,key and value not same because each head should learn differnt patterns

In [2]:

x = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

In [3]:
#Mathematical Flow
#head_i=Attention(Qi,Ki,Vi)
#then combine
#multihead(Q,K,V)=Concat(head1,head2,head3.....)

In [4]:
#Creating linear layers these generates Q,K,V
#why linear?
#they learn how to transform embeddings into  usefull attention representation

In [5]:
#for head1
WQ1=nn.Linear(4,4)
WK1=nn.Linear(4,4)
WV1=nn.Linear(4,4)

#for head2
WQ2=nn.Linear(4,4)
WK2=nn.Linear(4,4)
WV2=nn.Linear(4,4)

In [6]:
WQ1

Linear(in_features=4, out_features=4, bias=True)

In [7]:
#for head1
Q1=WQ1(x)
K1=WK1(x)
V1=WV1(x)

#for head2

Q2=WQ2(x)
K2=WK2(x)
V2=WV2(x)

In [8]:
Q1

tensor([[-0.1153,  0.5073,  0.8898, -0.5837],
        [ 0.1436,  0.3548,  0.8079, -0.7253],
        [-0.1185,  0.6227,  1.0935, -0.7137]], grad_fn=<AddmmBackward0>)

In [9]:
def attention(Q,K,V):
  scores=torch.matmul(Q,K.T)

  dk=K.shape[-1]

  scaled_scores=scores / torch.sqrt(torch.tensor(dk,dtype=torch.float32))

  attention_weights=F.softmax(scaled_scores,dim=1)

  output=torch.matmul(attention_weights,V)

  return output

In [10]:
head1=attention(Q1,K1,V1)
head2=attention(Q2,K2,V2)

In [11]:
head1

tensor([[-0.1192,  0.9856,  0.2905,  0.3285],
        [-0.1141,  0.9701,  0.3148,  0.3016],
        [-0.1211,  0.9990,  0.2636,  0.3558]], grad_fn=<MmBackward0>)

In [12]:
multi_head_output = torch.cat([head1, head2], dim=1)

print(multi_head_output)

tensor([[-0.1192,  0.9856,  0.2905,  0.3285,  0.2650,  0.6864, -0.7624, -1.1196],
        [-0.1141,  0.9701,  0.3148,  0.3016,  0.1669,  0.8220, -1.0203, -1.3342],
        [-0.1211,  0.9990,  0.2636,  0.3558,  0.2178,  0.7548, -0.8929, -1.2295]],
       grad_fn=<CatBackward0>)


In [13]:
# now output contains multiple types of understanding instead of single attention view

# **Positinal Encoding**


*   Transformers create unique wave pattern for every position like fingerprints
*   nearby positions have similar patters and far positions have different patters this helps model learn sequence structure




In [14]:
import torch
import math

In [15]:
seq_len = 5 # suppose sentence length=5
d_model = 6 # embedding size = 6

In [16]:
PE=torch.zeros(seq_len,d_model) #creating empty matrix
PE

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

In [17]:
PE.shape #5 positions, 6-dimensional embeddings

torch.Size([5, 6])

In [18]:
position=torch.arange(0,seq_len).unsqueeze(1)

In [19]:
torch.arange(0,seq_len).unsqueeze(1)

tensor([[0],
        [1],
        [2],
        [3],
        [4]])

In [20]:
#computer frequencies

div_term=torch.exp(
    torch.arange(0,d_model,2) *
    (-math.log(10000.0)/d_model)
)

In [21]:
torch.exp(
    torch.arange(0,d_model,2) *
    (-math.log(10000.0)/d_model)
)

tensor([1.0000, 0.0464, 0.0022])

In [22]:
torch.arange(0,d_model,2)

tensor([0, 2, 4])

In [23]:
(-math.log(10000.0)/d_model)

-1.5350567286626973

In [24]:
#Apply sin and cos
# from even columns use sin like tranformer paper
PE[:, 0::2] = torch.sin(position * div_term)

# from odd columns use cos like tranformer paper
PE[:, 1::2] = torch.cos(position * div_term)

In [25]:
PE[:, 0::2]

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086]])

In [26]:
print(PE)

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000]])


# **Complete Architecture of transformers**

In [27]:
import torch
import torch.nn as nn

In [28]:
class transformer(nn.Module):
  def __init__(self,embed_dim,num_heads):
    super().__init__()


    #Multi_head_attention
    self.attention=nn.MultiheadAttention(
        embed_dim=embed_dim,
        num_heads=num_heads,
        batch_first=True
    )

    #Layer Normalization
    self.norm1=nn.LayerNorm(embed_dim)


    #Feed Forward Network

    self.ff=nn.Sequential(
        nn.Linear(embed_dim,embed_dim * 4),
        nn.ReLU(),
        nn.Linear(embed_dim * 4,embed_dim)
    )

    #Layer Normalization
    self.norm2=nn.LayerNorm(embed_dim)

  def forward(self,x):

    # Self attention
    attention_output,_ = self.attention(x,x,x)

    # Residual connection + normalization

    x=self.norm1(x+attention_output)

    ff_output=self.ff(x)

    x=self.norm2(x + ff_output)

    return x







In [29]:

# Example input
x = torch.rand(1, 3, 8)



In [30]:
# Transformer block
block = transformer(embed_dim=8, num_heads=2)
# Forward pass
output = block(x)

In [31]:


print("Output Shape:")
print(output.shape)

print("\nTransformer Output:")
print(output)

Output Shape:
torch.Size([1, 3, 8])

Transformer Output:
tensor([[[-0.6825,  1.0754,  0.4431, -0.3970,  0.1962, -2.1863,  0.8103,
           0.7408],
         [ 1.0641, -0.2245,  0.2694, -1.0598, -0.4302, -0.5180,  1.9929,
          -1.0938],
         [ 0.4475, -0.5522, -0.1533,  0.2556, -1.5395, -1.1175,  0.9755,
           1.6839]]], grad_fn=<NativeLayerNormBackward0>)
